# День 1. От технического текста к числовым признакам

**Практический блокнот на 4 пары (4 × 90 минут).** Тема: базовый конвейер NLP на материалах по теории вероятностей и математической статистике.

## Результаты обучения
После работы студент умеет:
- очищать и токенизировать русскоязычные технические тексты;
- объяснять, почему предобработка должна учитывать математические обозначения;
- исследовать частоты терминов, словарь и n-граммы;
- строить представления Bag of Words и TF–IDF;
- понимать, что именно теряется при переходе от текста к вектору.

**Среда:** Python 3, Jupyter/Colab. Основные упражнения используют стандартную библиотеку; для TF–IDF нужен `scikit-learn`. В ячейках есть технические тексты, а не финансовые кейсы.

## Пара 1 из 4. Текст как данные: корпус, очистка и нормализация

### 1.1. Сформируем корпус
В корпусе есть формулировки задач, определения и интерпретации из теории вероятностей и матстатистики. Обратите внимание на числа, греческие буквы, знаки сравнения и сокращения: чрезмерная очистка может удалить полезный смысл.

In [ ]:
documents = [
    "Вероятность события A равна 0.35; найти вероятность противоположного события.",
    "Случайная величина X имеет биномиальное распределение: X ~ Bin(n, p).",
    "Для нормального распределения X ~ N(mu, sigma^2) найти P(X < a).",
    "Оценить математическое ожидание и дисперсию по выборке x1, x2, ..., xn.",
    "Построить доверительный интервал для математического ожидания при неизвестной дисперсии.",
    "Проверить нулевую гипотезу H0: mu = mu0 против альтернативы H1: mu != mu0.",
    "Для выборки вычислить медиану, квартили и межквартильный размах.",
    "Найти коэффициент корреляции Пирсона и проверить его статистическую значимость.",
    "Оценить параметры линейной регрессии методом наименьших квадратов.",
    "Для независимых испытаний Бернулли определить вероятность не менее k успехов."
]

for i, text in enumerate(documents, 1):
    print(f"{i:02d}. {text}")

### 1.2. Сравним исходный текст и нормализованную версию

Нормализация — не универсальная операция. Приведение к нижнему регистру обычно безопасно для тематического поиска, но может быть нежелательно для обозначений случайных величин и гипотез. Удаление пунктуации упрощает текст, однако способно разрушить записи вроде `H0: mu = mu0`, `p-value < 0.05` и `N(mu, sigma^2)`.

In [ ]:
import re
import unicodedata

def normalize_text(text):
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("ё", "е").replace("Ё", "Е")
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()

sample = "H0: μ = μ0; p-value < 0.05; X ~ N(μ, σ²)"
print("Исходный:", sample)
print("Нормализованный:", normalize_text(sample))

### 1.3. Сравним разные стратегии очистки

В первом варианте сохраняем математические символы и числа. Во втором оставляем только буквы. Сопоставьте результаты и решите, какой вариант лучше подходит для поиска задач по математической статистике.

In [ ]:
def keep_math_symbols(text):
    text = normalize_text(text)
    return re.findall(r"[a-zа-я0-9]+(?:[._-][a-zа-я0-9]+)*|[=<>~μσ²αβ]+", text)

def letters_only(text):
    return re.findall(r"[a-zа-я]+", normalize_text(text))

for text in [
    "P(X < a) = 0.95",
    "H0: mu = 0, p-value < 0.05",
    "X ~ N(mu, sigma^2)"
]:
    print("\n", text)
    print("С математическими знаками:", keep_math_symbols(text))
    print("Только буквы:", letters_only(text))

### Задания к паре 1

1. Почему строка `X < 2` после неосторожной очистки может потерять принципиальную часть условия?
2. Какие символы необходимо сохранить в коллекции задач по терверу?
3. Придумайте два правила очистки: одно для тематического поиска по формулировкам, второе — для извлечения математических условий.
4. Добавьте в корпус два собственных задания по комбинаторике или статистике.

**Мини-практика:** реализуйте функцию, которая нормализует пробелы, но сохраняет числа, знаки сравнения и латинские обозначения.

## Пара 2 из 4. Токенизация, регулярные выражения и n-граммы

### 2.1. Базовая токенизация

Токенизация разбивает строку на элементы, с которыми работает алгоритм. Простой регулярный шаблон подходит для первого эксперимента, но не является полноценным морфологическим анализатором и не всегда корректно обрабатывает математические записи.

In [ ]:
def tokenize_words(text):
    return re.findall(r"[а-яёa-z]+", text.lower())

for text in documents[:5]:
    print(tokenize_words(text))

### 2.2. Сохраним математические термины и обозначения

Для технических текстов можно использовать токенизатор, который распознаёт слова, десятичные числа, обозначения с дефисом и отдельные математические знаки. Это всё ещё упрощённая учебная эвристика.

In [ ]:
TOKEN_PATTERN = r"[a-zа-яё]+(?:-[a-zа-яё]+)*|\d+(?:[.,]\d+)?|[=<>~]+|[μσ²αβ]"

def tokenize_technical(text):
    return re.findall(TOKEN_PATTERN, text.lower())

examples = [
    "p-value < 0.05",
    "X ~ N(mu, sigma^2)",
    "H0: mu = mu0",
    "Метод Монте-Карло: 10 000 испытаний"
]
for text in examples:
    print(text, "->", tokenize_technical(text))

### 2.3. Слова, формы слов и леммы

Слова «выборка», «выборки», «выборкой» имеют близкий смысл, но при простом подсчёте считаются разными токенами. Лемматизация приводит словоформы к словарной форме; стемминг отсекает окончания и части слова по правилам. Эти методы различаются и могут ошибаться на терминах.

В следующем упражнении создадим небольшой словарь ручных замен — не как промышленное решение, а как демонстрацию влияния нормализации.

In [ ]:
forms = [
    "выборка", "выборки", "выборкой", "выборке",
    "гипотеза", "гипотезы", "гипотезой",
    "распределение", "распределения", "распределением"
]
print("Различных форм до нормализации:", len(set(forms)))

toy_lemma = {
    "выборки": "выборка", "выборкой": "выборка", "выборке": "выборка",
    "гипотезы": "гипотеза", "гипотезой": "гипотеза",
    "распределения": "распределение", "распределением": "распределение"
}
normalized_forms = [toy_lemma.get(w, w) for w in forms]
print("Различных форм после ручной нормализации:", len(set(normalized_forms)))
print(normalized_forms)

### 2.4. Биграммы и триграммы

Одиночные слова не всегда сохраняют терминологию. Сочетания «случайная величина», «нулевая гипотеза», «доверительный интервал» и «центральная предельная теорема» часто информативнее отдельных слов.

In [ ]:
from collections import Counter

tokens = [tokenize_words(doc) for doc in documents]

def ngrams(tokens, n=2):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

for doc_tokens in tokens[:4]:
    print("Биграммы:", ngrams(doc_tokens, 2))
    print("Триграммы:", ngrams(doc_tokens, 3))

### Задания к паре 2

1. Составьте регулярное выражение, которое извлекает десятичные числа, включая запись с запятой.
2. Почему выражение `X1` может быть полезнее сохранить целиком, чем разделить на `x` и `1`?
3. Выпишите пять биграмм, характерных для задач по математической статистике.
4. Сравните частоты слов «распределение» и «нормальное распределение». Какой термин точнее описывает тему задачи?

**Мини-практика:** измените токенизатор так, чтобы он сохранял сочетания вида `p-value` и числа вида `0.05`.

## Пара 3 из 4. Статистика корпуса и распределение частот

### 3.1. Частоты токенов и словарь

Исследование частот — первый этап анализа корпуса. Частые слова могут быть общими для многих документов, а редкие — указывать на конкретную тему. Размер корпуса, длины текстов и состав терминологии влияют на выводы.

In [ ]:
all_tokens = [token for doc in documents for token in tokenize_words(doc)]
freq = Counter(all_tokens)

print("Число документов:", len(documents))
print("Число токенов:", len(all_tokens))
print("Число уникальных токенов:", len(set(all_tokens)))
print("\nТоп-20 токенов:")
for word, count in freq.most_common(20):
    print(f"{word:20s} {count}")

### 3.2. Удаление стоп-слов: что меняется?

Стоп-слова — очень частотные служебные слова, которые иногда мало помогают тематическому поиску. Но слова «не», «без», «менее» могут менять смысл математического условия. Удалять их без проверки нельзя: фраза «гипотеза не отвергается» не равна фразе «гипотеза отвергается».

In [ ]:
stop_words = {"и", "для", "по", "при", "из", "на", "в", "равна", "найти"}

def remove_stop_words(tokens, stop_words):
    return [t for t in tokens if t not in stop_words]

test_sentence = "Нулевая гипотеза не отвергается при уровне значимости 0.05"
print("Исходные токены:", tokenize_words(test_sentence))
print("После удаления стоп-слов:",
      remove_stop_words(tokenize_words(test_sentence), stop_words))

### 3.3. Закон Ципфа и длинный хвост частот

В естественных языках небольшое число слов часто встречается очень часто, а большинство слов — редко. Это качественное наблюдение известно как закон Ципфа. На небольшом учебном корпусе нельзя делать строгие статистические выводы, но можно изучить форму распределения.

In [ ]:
ranked = freq.most_common()
print("Ранг | слово | частота")
for rank, (word, count) in enumerate(ranked[:25], start=1):
    print(f"{rank:4d} | {word:20s} | {count}")

### 3.4. Визуализация частот (необязательно)

Если установлен `matplotlib`, построим график частот наиболее распространённых слов. График помогает заметить разрыв между частыми общими словами и специализированными терминами.

In [ ]:
try:
    import matplotlib.pyplot as plt

    top = freq.most_common(15)
    labels_plot = [x[0] for x in top]
    values_plot = [x[1] for x in top]
    plt.figure(figsize=(11, 4))
    plt.bar(labels_plot, values_plot)
    plt.xticks(rotation=55, ha="right")
    plt.ylabel("Частота")
    plt.title("Частоты токенов в учебном корпусе")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("Для графика установите matplotlib: python -m pip install matplotlib")

### Задания к паре 3

1. Найдите три наиболее частотных слова и объясните, почему они часто встречаются.
2. Какие стоп-слова нельзя удалять в корпусе математических условий?
3. Добавьте 5 документов на одну тему и сравните частоты до и после добавления.
4. Почему сравнивать абсолютные частоты корпусов разного размера некорректно без нормировки?

**Мини-практика:** вычислите долю каждого слова в общем количестве токенов и сравните её с абсолютной частотой.

## Пара 4 из 4. Bag of Words, TF–IDF и представление документов

### 4.1. Bag of Words вручную

Bag of Words (BoW) описывает документ вектором частот слов фиксированного словаря. Порядок слов теряется: предложения с одинаковым набором слов могут получить одинаковый вектор, даже если смысл или логическая структура различаются.

In [ ]:
toy_docs = [
    "случайная величина имеет нормальное распределение",
    "нормальное распределение имеет случайная величина",
    "нулевая гипотеза проверяется по выборке",
    "доверительный интервал оценивается по выборке"
]
toy_tokens = [tokenize_words(doc) for doc in toy_docs]
vocabulary = sorted(set(word for doc in toy_tokens for word in doc))

def bow_vector(tokens, vocabulary):
    counts = Counter(tokens)
    return [counts[word] for word in vocabulary]

print("Словарь:", vocabulary)
for doc, tokens_doc in zip(toy_docs, toy_tokens):
    print(doc)
    print(bow_vector(tokens_doc, vocabulary))

### 4.2. TF–IDF

TF–IDF сочетает частоту термина в документе (TF) с обратной частотой документов (IDF). Термин, встречающийся во многих документах, обычно получает меньший IDF-вес, чем термин, характерный для небольшого числа документов. Конкретная формула и сглаживание зависят от реализации.

In [ ]:
try:
    from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

    count_vectorizer = CountVectorizer(token_pattern=r"(?u)\b[а-яёА-ЯЁa-zA-Z]+\b")
    X_count = count_vectorizer.fit_transform(documents)
    tfidf_vectorizer = TfidfVectorizer(
        token_pattern=r"(?u)\b[а-яёА-ЯЁa-zA-Z]+\b",
        ngram_range=(1, 2)
    )
    X_tfidf = tfidf_vectorizer.fit_transform(documents)

    print("BoW: ", X_count.shape, "— документы × признаки")
    print("TF-IDF:", X_tfidf.shape, "— документы × признаки")
    print("Первые 20 признаков:", tfidf_vectorizer.get_feature_names_out()[:20])
except ImportError:
    print("Установите scikit-learn: python -m pip install scikit-learn")

### 4.3. Посмотрим на самые значимые признаки документа

Высокий TF–IDF-вес показывает, что признак характерен для документа относительно данного корпуса. Это не означает, что слово объективно важно во всех контекстах.

In [ ]:
try:
    feature_names = tfidf_vectorizer.get_feature_names_out()
    row = X_tfidf[4].toarray().ravel()
    top_idx = row.argsort()[::-1][:10]
    print("Документ:", documents[4])
    print("Наиболее весомые признаки:")
    for idx in top_idx:
        if row[idx] > 0:
            print(f"{feature_names[idx]:25s} {row[idx]:.3f}")
except (NameError, ImportError):
    print("Сначала выполните ячейку с построением TF–IDF.")

### Итоговая работа дня

**Задача:** создать мини-корпус из 15–20 формулировок по одной из тем:
- комбинаторика и классическая вероятность;
- распределения случайных величин;
- выборочные характеристики;
- доверительные интервалы;
- проверка статистических гипотез.

Выполните этапы:
1. сохраните тексты в список;
2. выберите и обоснуйте правила токенизации;
3. посчитайте частоты и оцените влияние стоп-слов;
4. сравните BoW и TF–IDF;
5. запишите три ограничения выбранного представления.

### Контрольные вопросы
- Почему предобработка зависит от предметной области?
- В чём разница между токеном, типом токена и n-граммой?
- Что означает разреженность матрицы BoW?
- Почему TF–IDF не понимает математическую эквивалентность двух формулировок?
- Какие признаки математической задачи нельзя бездумно удалять?